In [1]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 步骤1：转小写，去除标点，只保留字母和空格
    text_lower = text.lower()
    # 生成标点替换表，删除所有标点
    punc_table = str.maketrans('', '', string.punctuation)
    text_clean = text_lower.translate(punc_table)
    
    # 步骤2：按空格分词（自动处理多空格）
    words = text_clean.split()
    
    # 步骤3：构建词汇表，按出现频率降序，ID从0开始
    word_count = Counter(words)
    # 按频次从高到低排序，频次相同保持原出现顺序
    sorted_words = sorted(word_count.keys(), key=lambda x: (-word_count[x], words.index(x)))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 步骤4：滑动窗口生成长度n的特征序列与对应标签
    features = []
    labels = []
    total_len = len(words)
    # 滑动窗口遍历，窗口结束位置不能超过总词数-1（保证有下一个词）
    for i in range(total_len - n):
        window = words[i:i+n]
        next_word = words[i + n]
        features.append(window)
        labels.append(next_word)
    
    return vocab, (features, labels)

# 测试示例
if __name__ == "__main__":
    test_str = "The time machine"
    vocab, (feat, lab) = preprocess_text(test_str, n=2)
    print("词汇表：", vocab)
    print("特征列表：", feat)
    print("标签列表：", lab)

词汇表： {'the': 0, 'time': 1, 'machine': 2}
特征列表： [['the', 'time']]
标签列表： ['machine']


In [2]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播
    参数：
        x_t: (batch_size, input_size)
        h_prev: (batch_size, hidden_size)
        W_hx: (input_size, hidden_size)
        W_hh: (hidden_size, hidden_size)
        b_h: (1, hidden_size)
    返回：
        h_t: 当前隐藏状态 (batch, hidden)
        cache: 缓存前向中间变量，用于反向传播
    """
    z_t = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    h_t = np.tanh(z_t)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, z_t, h_t)
    return h_t, cache


def rnn_backward(dh_next, cache):
    """
    RNN单步反向传播，计算各参数梯度
    参数：
        dh_next: 损失对h_t的梯度 (batch, hidden_size)
        cache: 前向传播保存的中间变量
    返回：
        dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x_t, h_prev, W_hx, W_hh, b_h, z_t, h_t = cache
    batch_size = x_t.shape[0]
    
    # 1. 求dz_t
    dz_t = dh_next * (1 - np.square(h_t))  # 逐元素相乘
    
    # 2. 各权重梯度
    dW_hx = np.dot(x_t.T, dz_t)
    dW_hh = np.dot(h_prev.T, dz_t)
    db_h = np.sum(dz_t, axis=0, keepdims=True)  # 沿batch求和
    
    # 3. 输入与上一隐状态梯度
    dx_t = np.dot(dz_t, W_hx.T)
    dh_prev = np.dot(dz_t, W_hh.T)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ---------------------- 测试示例 ----------------------
if __name__ == "__main__":
    # 超参数
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 随机初始化输入、隐状态、权重
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(1, hidden_size)
    
    # 前向传播
    h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("===== 前向传播结果 =====")
    print("h_t shape:", h_t.shape)  # (2,4)
    
    # 模拟上游梯度dh_next
    dh_next = np.random.randn(batch_size, hidden_size)
    # 反向传播求梯度
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)
    
    print("\n===== 反向传播梯度维度校验 =====")
    print(f"dx_t shape: {dx_t.shape} (预期 {batch_size, input_size})")
    print(f"dh_prev shape: {dh_prev.shape} (预期 {batch_size, hidden_size})")
    print(f"dW_hx shape: {dW_hx.shape} (预期 {input_size, hidden_size})")
    print(f"dW_hh shape: {dW_hh.shape} (预期 {hidden_size, hidden_size})")
    print(f"db_h shape: {db_h.shape} (预期 (1, {hidden_size}))")

===== 前向传播结果 =====
h_t shape: (2, 4)

===== 反向传播梯度维度校验 =====
dx_t shape: (2, 3) (预期 (2, 3))
dh_prev shape: (2, 4) (预期 (2, 4))
dW_hx shape: (3, 4) (预期 (3, 4))
dW_hh shape: (4, 4) (预期 (4, 4))
db_h shape: (1, 4) (预期 (1, 4))


In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        # 双向单层RNN，输入shape(seq_len, batch, input_dim)
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
            num_layers=1,
            batch_first=False  # 匹配输入seq_len在前的格式
        )

    def forward(self, X):
        """
        参数：
            X: 输入序列 (seq_len, batch, input_dim)
        返回：
            outputs: 各时间步拼接隐状态 (seq_len, batch, 2*hidden_dim)
            seq_repr: 全局序列表示 (batch, 2*hidden_dim)
        """
        # out: (seq_len, batch, 2*hidden_dim) 每一步前向、后向隐状态拼接
        # h_n: (num_layers * 2, batch, hidden_dim)
        out, h_n = self.rnn(X)

        # h_n拆分：[前向最后隐状态, 后向最后隐状态]
        h_forward_last = h_n[0]   # (batch, hidden_dim)
        h_backward_first = h_n[1] # (batch, hidden_dim)

        # 全局序列表示：拼接前向末尾 + 后向开头
        seq_repr = torch.cat([h_forward_last, h_backward_first], dim=-1)

        return out, seq_repr


# ---------------- 测试代码 ----------------
if __name__ == "__main__":
    # 超参数
    seq_len = 10
    batch = 4
    input_dim = 8
    hidden_dim = 16

    # 构造输入 (seq_len, batch, input_dim)
    X = torch.randn(seq_len, batch, input_dim)
    encoder = BiRNNEncoder(input_dim, hidden_dim)

    outputs, seq_repr = encoder(X)

    print("各时间步拼接隐状态 shape:", outputs.shape)
    # 预期 torch.Size([10, 4, 32])  2*16=32
    print("全局序列表示 shape:", seq_repr.shape)
    # 预期 torch.Size([4, 32])

各时间步拼接隐状态 shape: torch.Size([10, 4, 32])
全局序列表示 shape: torch.Size([4, 32])


In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_batch, target_batch, W, W_out):
    """
    CBOW前向传播 + 完整softmax交叉熵损失计算（无负采样）
    参数：
        context_batch: 上下文索引，shape [batch, context_size]
        target_batch: 中心词索引，shape [batch]
        W: 输入嵌入矩阵 (V, d)
        W_out: 输出权重矩阵 (d, V)
    返回：
        loss: batch平均交叉熵损失（标量tensor）
    """
    batch_size, context_size = context_batch.shape
    
    # 1. 查表获取所有上下文词向量 [batch, context_size, d]
    context_embeds = W[context_batch]
    
    # 2. 上下文向量求平均，得到隐藏层 h [batch, d]
    h = torch.mean(context_embeds, dim=1)
    
    # 3. 计算词汇得分 logits [batch, V]
    logits = torch.matmul(h, W_out)
    
    # 4. 完整softmax + 交叉熵损失（内置log_softmax+nll_loss等价手动计算）
    loss = F.cross_entropy(logits, target_batch)
    
    return loss


# ---------------- 测试示例 ----------------
if __name__ == "__main__":
    # 超参数
    V = 100    # 词汇表大小
    d = 16     # 嵌入维度
    batch = 8  # 批次大小
    context_size = 4  # 单侧窗口1，左右共4个上下文词
    
    # 随机初始化权重矩阵
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    
    # 构造模拟输入
    context_batch = torch.randint(0, V, (batch, context_size))
    target_batch = torch.randint(0, V, (batch,))
    
    # 前向计算损失
    loss_val = cbow_forward_loss(context_batch, target_batch, W, W_out)
    print(f"CBOW batch平均损失值: {loss_val.item():.4f}")

CBOW batch平均损失值: 5.4451


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FixedMultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # 题目固定参数
        self.num_heads = 2
        self.d_model = 4
        self.d_k = self.d_model // self.num_heads  # d_k = 2
        
        # Q/K/V 投影层：d_model -> d_model（均分至2个头，每个头2维）
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        
        # 多头拼接后的最终输出线性层
        self.w_o = nn.Linear(self.d_model, self.d_model)

    def scaled_dot_product_attn(self, q, k, v):
        """缩放点积注意力（单头）
        q/k/v shape: (num_heads, seq_len, batch, d_k)
        """
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn_weight = F.softmax(attn_score, dim=-1)
        attn_out = torch.matmul(attn_weight, v)
        return attn_out

    def forward(self, X):
        """
        输入 X: (seq_len, batch, d_model)
        返回 output: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影 Q,K,V
        Q = self.w_q(X)  # (seq_len, batch, 4)
        K = self.w_k(X)
        V = self.w_v(X)
        
        # 2. 分头：拆分维度到 num_heads，调换维度方便多头计算
        # (seq_len, batch, num_heads, d_k) -> (num_heads, seq_len, batch, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 0, 1, 3)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 0, 1, 3)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 0, 1, 3)
        
        # 3. 每个头独立计算缩放点积注意力
        head_out = self.scaled_dot_product_attn(Q, K, V)  # (num_heads, seq_len, batch, d_k)
        
        # 4. 拼接所有头：恢复维度顺序，合并多头维度
        head_out = head_out.permute(1, 2, 0, 3).contiguous()  # (seq_len, batch, num_heads, d_k)
        concat_out = head_out.view(seq_len, batch, self.d_model)  # 2*2=4
        
        # 5. 最终线性层输出
        output = self.w_o(concat_out)
        return output


# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    # 模拟输入：seq_len=5, batch=3, d_model=4
    seq_len = 5
    batch_size = 3
    X = torch.randn(seq_len, batch_size, 4)
    
    mha = FixedMultiHeadAttention()
    out = mha(X)
    
    print("输入形状:", X.shape)
    print("输出形状:", out.shape)
    # 输出预期：torch.Size([5, 3, 4])，和输入维度完全一致

输入形状: torch.Size([5, 3, 4])
输出形状: torch.Size([5, 3, 4])
